In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Nhập đường dẫn thư mục dự án trên Drive của bạn
default_path = '/content/drive/MyDrive/arXiv'
project_dir = (
    input(f'Thư mục dự án trên Drive [{default_path}]: ').strip()
    or default_path
)
project_root = Path(project_dir).expanduser()
project_root.mkdir(parents=True, exist_ok=True)

os.chdir(project_root)
print(f'Thư mục làm việc hiện tại: {Path.cwd()}')

In [ ]:
from pathlib import Path
import subprocess, sys

repo_dir = Path('code')
repo_url = 'https://github.com/NewB-S2deeC0de/arXiv.git'

if not repo_dir.exists():
  print('Đang clone repo mới từ GitHub...')
  subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
else:
  print('Đang cập nhật repo từ GitHub...')
  subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)

print('Cài đặt dependencies...')
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'src/requirements.txt'),
    ],
    check=True,
)

# Thêm module crawler vào sys.path
src_path = str((repo_dir / 'src').resolve())
if src_path not in sys.path:
  sys.path.insert(0, src_path)

# Tạo thư mục đĩa tạm SSD cục bộ để tải và bóc tách source tar.gz
scratch_dir = Path('/tmp/crawler_scratch')
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f'Ổ đĩa tạm SSD sẵn sàng tại: {scratch_dir}')

In [ ]:
from crawler.config import load_config
from crawler.secrets import load_semantic_scholar_key

config_path = repo_dir / 'config/crawler.yaml'
config = load_config(config_path)

# Đảm bảo scratch_root trỏ vào SSD /tmp để đạt tốc độ cao nhất
config.paths['scratch_root'] = '/tmp/crawler_scratch'

s2_key = load_semantic_scholar_key()
if s2_key:
  print('Semantic Scholar: Authentication = KEY (Đã nhận diện API key)')
else:
  print('CẢNH BÁO: Chưa tìm thấy SEMANTIC_SCHOLAR_API_KEY trong Colab Secrets!')
  print(
      '-> Đang ở chế độ anonymous. Hãy thêm API key vào Colab Secrets để tránh'
      ' bị Cloudflare chặn IP.'
  )

rng = config.range
print(
    f"Phạm vi cào dữ liệu: {rng['start_month']}.{rng['start_id']:05d} đến"
    f" {rng['end_month']}.{rng['end_id']:05d}"
)

In [ ]:
from crawler.progress import ProgressStore
import pandas as pd

store = ProgressStore(config.paths['persistent_root'])
counts = store.status_counts()

# Hiển thị bảng tổng hợp tiến độ dạng DataFrame
df_status = pd.DataFrame(counts).fillna(0).astype(int)
print('=== TIẾN ĐỘ THU THẬP DỮ LIỆU HIỆN TẠI ===')
display(df_status)

In [ ]:
from crawler.pipeline import run_pipeline

# Đổi thành True nếu phiên trước bị ngắt đột ngột và dính file lock
TAKEOVER_STALE = False

print('Bắt đầu tiến trình cào toàn bộ 5.000 paper...')
summary = await run_pipeline(
    config=config,
    stages={'arxiv', 's2'},
    resume=True,
    limit=None,  # limit=None để cào trọn vẹn 5.000 IDs
    takeover_stale=TAKEOVER_STALE,
)

print('\n=== HOÀN TẤT PHIÊN CHẠY ===')
for k, v in summary.items():
  if k != 'events':
    print(f'{k}: {v}')

In [ ]:
# Chỉ đổi thành True khi bạn muốn thử lại các bài bị lỗi mạng hoặc 5xx
RETRY_FAILED = False

if RETRY_FAILED:
  import uuid
  from crawler.locking import DatasetLock

  print('Đang reset các task bị failed để chuẩn bị retry...')
  with DatasetLock(
      config.paths['persistent_root'],
      'retry-' + uuid.uuid4().hex,
      takeover_stale=True,
  ):
    store = ProgressStore(config.paths['persistent_root'])
    store.reset(
        'failed', stages={'source', 'metadata', 'references', 'discovery'}
    )

  print('Đang chạy lại các task lỗi...')
  summary = await run_pipeline(
      config, {'arxiv', 's2'}, resume=True, limit=None, takeover_stale=True
  )
  print(summary)
else:
  print('Bỏ qua bước retry (RETRY_FAILED=False).')

In [ ]:
import uuid
from crawler.export import export_snapshot

run_id = 'full-5000-' + uuid.uuid4().hex[:8]
print(f'Đang xuất snapshot dataset với Run ID: {run_id}...')
manifest = export_snapshot(config, run_id)

print('\n=== XUẤT DATASET THÀNH CÔNG ===')
print(f"Tổng số bài báo (papers.parquet): {manifest['papers']}")
print(f"Tổng số liên kết trích dẫn (references.parquet): {manifest['references']}")
print(f"Đường dẫn thư mục xuất: {config.paths['persistent_root']}/exports/{run_id}")

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

export_dir = Path(config.paths['persistent_root']) / 'exports' / run_id

df_papers = pq.read_table(export_dir / 'papers.parquet').to_pandas()
df_refs = pq.read_table(export_dir / 'references.parquet').to_pandas()

print(f'=== DỮ LIỆU BẢNG PAPERS ({len(df_papers)} dòng) ===')
display(df_papers.head(5))

print(f'\n=== DỮ LIỆU BẢNG REFERENCES ({len(df_refs)} dòng) ===')
display(df_refs.head(5))